In [31]:
# Cloning GitHub repo
!git clone https://github.com/Romit-M/UIDAI-Hackathon-2026.git
%cd UIDAI-Hackathon-2026

# IMPORTS
!pip install rapidfuzz -q
from rapidfuzz import process, utils

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# HELPER FUNCTIONS

# Data loader
def load_data(file_path):
  df = pd.read_csv(file_path)
  df.columns = df.columns.str.lower()
  return df

# Getting file names
def get_filename(category, r):
  return f"api_data_aadhar_{category}_{r}.csv"


Cloning into 'UIDAI-Hackathon-2026'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 64 (delta 5), reused 7 (delta 3), pack-reused 51 (from 1)
Receiving objects: 100% (64/64), 82.54 MiB | 10.14 MiB/s, done.
Resolving deltas: 100% (21/21), done.
Updating files: 100% (24/24), done.
/content/UIDAI-Hackathon-2026


### **DATA CLEANING**

In [32]:
# TEXT NORMALIZATION (making text comparable)

def normalize_text(s):
  return(
      s.lower()
       .strip()
       .replace('&', 'and')
       .replace(' ', '')
  )


In [33]:
# UNIQUE STATE MAPPING USING FUZZY LOGIC

# Manual override for historic name changes
manual_fixes = {
    'orissa': 'odisha',
    'pondicherry': 'puducherry',
    'uttaranchal': 'uttarakhand',
    'damananddiu': 'dadraandnagarhavelianddamananddiu',
    'dadraandnagarhaveli': 'dadraandnagarhavelianddamananddiu'
}

# Official Aadhaar/UIDAI standard list
official_states = [
    'haryana', 'bihar', 'jammuandkashmir', 'tamilnadu', 'maharashtra',
    'gujarat', 'odisha', 'westbengal', 'kerala', 'rajasthan', 'punjab',
    'himachalpradesh', 'uttarpradesh', 'assam', 'uttarakhand',
    'madhyapradesh', 'karnataka', 'andhrapradesh', 'telangana', 'goa',
    'nagaland', 'jharkhand', 'delhi', 'chhattisgarh', 'meghalaya',
    'chandigarh', 'puducherry', 'manipur', 'sikkim', 'tripura',
    'mizoram', 'arunachalpradesh', 'ladakh', 'lakshadweep',
    'dadraandnagarhavelianddamananddiu', 'andamanandnicobarislands'
]

def state_mapping(df):

  # Manual override
  df['state'] = df['state'].replace(manual_fixes)

  # Find the closest match
  def get_closest_match(state):
      # Returns the best match from official_states if score > 80%
      match = process.extractOne(state, official_states, score_cutoff=80)
      return match[0] if match else state

  # Get unique dirty states
  dirty_states = df['state'].unique()

  # Create and apply an automated mapping dictionary
  auto_mapping = {ds: get_closest_match(ds) for ds in dirty_states}
  df['state'] = df['state'].map(auto_mapping)

  return df


In [34]:
# CLEANING PIPELINE

def clean_pipeline(df):
  df['state'] = df['state'].astype(str).apply(normalize_text)
  df['district'] = df['district'].astype(str).apply(normalize_text)

  df = state_mapping(df)

  return df


In [35]:
# DATA LOADING -> CLEANING -> SAVING CLEANED DATA

categories = ['biometric', 'demographic', 'enrolment']
ranges = ['0_500000', '500000_1000000']

RAW_PATH = "data/raw/api_data_aadhar"
PROCESSED_PATH = "data/processed/api_data_aadhar"

for category in categories:
  for r in ranges:

    # Load file
    filename = get_filename(category, r)
    file_path = f"{RAW_PATH}_{category}/{filename}"
    df = load_data(file_path)

    # Apply cleaning
    df = clean_pipeline(df)

    # Save cleaned file
    output_filename = filename.replace('.csv', '_cleaned.csv')
    output_path = f"{PROCESSED_PATH}_{category}/{output_filename}"
    df.to_csv(output_path, index=False)

    print(f"{output_filename} saved.")


api_data_aadhar_biometric_0_500000_cleaned.csv saved.
api_data_aadhar_biometric_500000_1000000_cleaned.csv saved.
api_data_aadhar_demographic_0_500000_cleaned.csv saved.
api_data_aadhar_demographic_500000_1000000_cleaned.csv saved.
api_data_aadhar_enrolment_0_500000_cleaned.csv saved.
api_data_aadhar_enrolment_500000_1000000_cleaned.csv saved.


In [36]:
# PUSH CLEANED FILES TO REPO

from google.colab import userdata
pat = userdata.get('GitHubAccessToken')

!git config --global user.name "Romit-M"
!git config --global user.email "romitrmaity@gmail.com"

!git add .
!git commit -m "Added cleaned data files"
!git push https://{pat}@github.com/Romit-M/UIDAI-Hackathon-2026.git


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


In [30]:
# !rm -rf /content/UIDAI-Hackathon-2026
# %cd /content


/content
